---

## **Projet Deep Learning**
## **Classification de tissus cancéreux colorectaux**

---


---

**Ce notebook est conçu pour être :**
- **reproductible** (chemins relatifs, seeds fixées)
- **idempotent** (relançable sans retélécharger si les fichiers sont déjà présents)
- **traçable** (quality gates go/no-go explicites)

---


---

# PARTIE 7 : COMPARAISON FINALE ET ANALYSE CRITIQUE

---


---

### Objectif

Comparer les 4 architectures entraînées (MLP, CNN, ResNet-18, ViT) sur les mêmes données, analyser les compromis (performance vs complexité vs temps), et répondre aux questions finales.

### Progression du projet
- **Partie 2 (MLP)** : baseline — pixels isolés
- **Partie 3 (CNN)** : structure spatiale locale
- **Partie 4 (ResNet-18)** : transfer learning
- **Partie 5 (ViT)** : attention globale

### Consignes du sujet
- Tableau comparatif : accuracy, nb paramètres, temps d'entraînement
- **Q7.1** : Quel modèle a le meilleur test accuracy ? Pourquoi ?
- **Q7.2** : Quel modèle recommander pour un déploiement clinique ? Justifier
- **Q7.3** : Un collègue suggère du random label smoothing (remplacer les one-hot par des vecteurs aléatoires uniformes). Sans lancer l'expérience, expliquer si ça aiderait ou nuirait

---

### Capacité de mémorisation vs efficacité des paramètres

| Modèle | Paramètres | Capacité de mémorisation | Test accuracy |
|--------|-----------|------------------------|--------------|
| MLP | 1 370 121 | Élevée — beaucoup de paramètres pour une tâche simple | 68.02% |
| CNN | ~280 000 | Modérée — filtres partagés (weight sharing) | à confirmer |
| ResNet | 11 181 129 | Très élevée — compensée par les poids pré-entraînés | 91.53% |
| ViT | ~870 000 | Élevée — mécanisme d'attention coûteux en paramètres | 78.87% |

Le MLP a 5× plus de paramètres que le CNN mais fait bien moins bien. C'est parce que les paramètres du CNN sont plus **efficaces** : un filtre 3×3 est réutilisé sur toute l'image (weight sharing), alors qu'une couche dense a un poids différent pour chaque pixel. Plus de paramètres ne signifie pas meilleur modèle — c'est la qualité de l'architecture qui compte.

### Méthodes de régularisation appliquées

| Méthode | MLP | CNN | ResNet | ViT | Rôle |
|---------|-----|-----|--------|-----|------|
| Dropout | ✓ (0.3) | ✓ (0.25→0.5) | ✓ | ✓ (0.1) | Désactive des neurones aléatoirement |
| BatchNorm | ✗ | ✓ | ✓ | ✗ | Normalise les activations entre couches |
| Weight decay (L2) | ✓ (1e-4) | ✓ (1e-4) | ✓ (1e-4) | ✓ (1e-4) | Pénalise les poids trop grands |
| Best checkpoint | ✓ | ✓ | ✓ | ✓ | Garde le modèle avant surapprentissage |
| Data augmentation | ✗ | ✓ | ✗ | ✗ | Crée des variantes d'images |
| Scheduler (CosineAnnealing) | ✓ | ✓ | ✓ | ✓ | Baisse le lr progressivement |

---



In [ ]:
# Imports
import sys
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import seaborn as sns

notebook_start_time = time.time()

print("Imports OK")


In [ ]:
# Tableau comparatif — résultats de toutes les parties
# Les valeurs sont issues des notebooks 2 à 5 (à mettre à jour après chaque run)

results = {
    'MLP': {
        'test_acc': 0.630780,
        'val_acc': 0.701,
        'n_params': 1370121,
        'train_time': 0,  # à remplir
        'epochs': 50,
        'best_epoch': 19,
        'biggest_confusion': 'Cancer -> Mucosa (356)',
        'worst_f1_class': 'Debris (0.42)',
    },
    'CNN (sans aug)': {
        'test_acc': 0.888579,
        'val_acc': 0.987,
        'n_params': 280000,  # à vérifier
        'train_time': 0,  # à remplir
        'epochs': 40,
        'best_epoch': 31,
        'biggest_confusion': 'Stroma -> Debris (103)',
        'worst_f1_class': 'Stroma (0.63)',
    },
    'ResNet-18 (frozen)': {
        'test_acc': 0.869916,
        'val_acc': 0.901,
        'n_params': 11181129,
        'train_time': 0,  # à remplir
        'epochs': 20,
        'best_epoch': 20,
        'biggest_confusion': '',
        'worst_f1_class': '',
    },
    'ResNet-18 (fine-tune)': {
        'test_acc': 0.915320,
        'val_acc': 0.993,
        'n_params': 11181129,
        'train_time': 0,  # à remplir
        'epochs': 20,
        'best_epoch': 19,
        'biggest_confusion': 'Stroma -> Debris (80)',
        'worst_f1_class': 'Stroma (0.67)',
    },
    'ViT (patch 7)': {
        'test_acc': 0.788719,
        'val_acc': 0.861,
        'n_params': 0,  # à remplir
        'train_time': 0,  # à remplir
        'epochs': 50,
        'best_epoch': 48,
        'biggest_confusion': '',
        'worst_f1_class': '',
    },
}

# Affichage du tableau
print("=" * 100)
print("TABLEAU COMPARATIF — TOUTES LES ARCHITECTURES")
print("=" * 100)
print(f"{'Modèle':<25s} {'Test acc':>10s} {'Val acc':>10s} {'Params':>12s} {'Époques':>8s} {'Best ep':>8s}")
print("-" * 75)
for name, r in results.items():
    print(f"{name:<25s} {r['test_acc']:>10.6f} {r['val_acc']:>10.3f} {r['n_params']:>12,} {r['epochs']:>8d} {r['best_epoch']:>8d}")

print(f"\n{'Modèle':<25s} {'Plus grande confusion':<35s} {'Pire F1':<20s}")
print("-" * 80)
for name, r in results.items():
    print(f"{name:<25s} {r['biggest_confusion']:<35s} {r['worst_f1_class']:<20s}")


In [ ]:
# Graphique comparatif — test accuracy
models = list(results.keys())
test_accs = [results[m]['test_acc'] * 100 for m in models]

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6', '#f39c12']
bars = ax.bar(models, test_accs, color=colors)
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Comparaison des architectures — Test Accuracy')
ax.set_ylim(50, 100)

# Afficher les valeurs sur les barres
for bar, acc in zip(bars, test_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{acc:.2f}%', ha='center', fontsize=9)

# Ligne seuil
ax.axhline(y=55, color='red', linestyle='--', alpha=0.5, label='Seuil MLP (55%)')
ax.axhline(y=75, color='orange', linestyle='--', alpha=0.5, label='Seuil CNN (75%)')
ax.axhline(y=85, color='green', linestyle='--', alpha=0.5, label='Seuil ResNet (85%)')
ax.legend()

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Compromis : nombre de paramètres vs test accuracy
fig, ax = plt.subplots(figsize=(10, 6))

for name, r in results.items():
    ax.scatter(r['n_params'], r['test_acc'] * 100, s=100, zorder=5)
    ax.annotate(name, (r['n_params'], r['test_acc'] * 100),
                textcoords="offset points", xytext=(10, 5), fontsize=8)

ax.set_xlabel('Nombre de paramètres')
ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Compromis : complexité du modèle vs performance')
ax.set_xscale('log')
plt.tight_layout()
plt.show()


---

### Q7.1 — Quel modèle a le meilleur test accuracy ? Pourquoi ?

*(À compléter avec les résultats définitifs après les runs)*

Le ResNet-18 fine-tuning obtient le meilleur test accuracy (~91.5%). Cela s'explique par le transfer learning : le modèle arrive avec des features visuelles pré-apprises sur 1.2M d'images (contours, textures, formes) et les adapte à l'histologie pendant le fine-tuning.

Le CNN from scratch est proche (~89%) — il apprend les features directement sur nos données, sans a priori. Sa performance est remarquable pour un modèle 40× plus léger que le ResNet.

Le ViT est le moins performant (~79%) car les Transformers ont besoin de beaucoup plus de données pour bien apprendre (le ViT original utilisait 300M images). Avec seulement 90K images et pas de pré-entraînement, il ne peut pas rivaliser.

Le MLP est la baseline (~63%) — il ne voit pas la structure spatiale des images, ce qui limite ses capacités sur de la classification d'images.

---


---

### Q7.2 — Quel modèle recommander pour un déploiement clinique ?

En contexte clinique, plusieurs critères comptent au-delà de l'accuracy :

**Le CNN from scratch** est recommandé pour les raisons suivantes :
- **Performance solide** (~89%) — proche du ResNet avec 40× moins de paramètres
- **Rapidité d'inférence** — images 28×28 traitées directement, pas de resize à 224×224
- **Légèreté** — ~280K paramètres vs 11.1M pour le ResNet, déployable sur des machines modestes
- **Pas de dépendance externe** — entraîné from scratch, pas de poids ImageNet nécessaires
- **Temps d'entraînement raisonnable** — ~9 min vs 65 min pour le ResNet fine-tuning

Le ResNet fine-tuning serait recommandé si l'accuracy maximale prime sur tout le reste (ex : dépistage où chaque point compte). Mais son coût en ressources (mémoire, temps, resize 8×) est un frein pour un déploiement à grande échelle.

Le ViT n'est pas adapté en l'état — il faudrait un pré-entraînement sur un large corpus d'images médicales.

**Point critique pour le clinique :** quel que soit le modèle, le stroma reste la classe la plus difficile (F1 ~0.63-0.67). Un système clinique devrait signaler les prédictions "stroma" comme incertaines et les envoyer à un pathologiste pour vérification.

---


---

### Q7.3 — Random label smoothing : aide ou nuit ?

Un collègue suggère de remplacer les labels one-hot par des vecteurs aléatoires uniformes pour améliorer la généralisation.

**Cela nuirait, et voici pourquoi :**

Le label smoothing **classique** (remplacer [1, 0, 0, ...] par [0.9, 0.01, 0.01, ...]) est une technique utile qui empêche le modèle d'être trop confiant dans ses prédictions. Ça fonctionne parce que la distribution reste informative : la vraie classe a toujours la plus haute probabilité.

Le **random** label smoothing (vecteurs aléatoires uniformes) détruit cette information : le modèle reçoit des labels qui ne correspondent plus aux images. C'est comme entraîner un enfant à reconnaître des animaux en lui disant au hasard "c'est un chat" ou "c'est un chien" — il n'apprendrait rien.

Concrètement :
- La loss ne convergerait pas car le gradient pointerait dans des directions aléatoires
- Le modèle apprendrait du bruit au lieu de patterns visuels
- L'accuracy plafonnerait autour du hasard (~11% pour 9 classes)

Le label smoothing classique (non aléatoire) aurait pu être bénéfique, notamment pour réduire les confusions Mucosa/Cancer où le modèle est souvent trop confiant.

---


---

## Bilan final du projet

### Ce qu'on a appris

| Architecture | Apport principal | Limite principale |
|-------------|-----------------|-------------------|
| MLP | Baseline simple et rapide | Ne voit pas la structure spatiale |
| CNN | Exploite les textures locales | Limité au contexte local (filtres 3×3) |
| ResNet-18 | Transfer learning puissant | Upscaling 8× dégrade les images |
| ViT | Attention globale | Besoin de beaucoup de données |

### Le domain shift : le vrai défi

L'écart val/test (~6-8 points selon les modèles) est constant quelle que soit l'architecture. Ce n'est pas un problème de modèle mais de données : les images test viennent d'un hôpital différent. Aucune architecture ne résout ce problème — il faudrait des données multi-hôpitaux ou des techniques de domain adaptation.

### Le stroma : le point faible partagé

Tous les modèles peinent sur le cancer-associated stroma (F1 ~0.45-0.67). Ce tissu conjonctif sans structure marquée est visuellement proche des débris et du smooth muscle. C'est une limite fondamentale à 28×28 pixels — la résolution est trop faible pour distinguer les fibres.

### Augmentations : la surprise

Contrairement à l'intuition, les augmentations n'ont pas amélioré les performances. Le dataset est suffisamment grand (90K images) pour que le CNN apprenne sans variantes artificielles. Les augmentations de couleur ont même nui en brouillant les nuances de teinte H&E.

---


---

## BONUS — Pistes d'amélioration pour le déploiement clinique

### Objectif clinique : minimiser les faux négatifs cancer

En contexte clinique, un faux négatif (cancer non détecté) est beaucoup plus grave qu'un faux positif (tissu sain classé cancer — revu par un pathologiste). Il faut donc maximiser le **recall** de la classe cancer, quitte à sacrifier de la précision.

### Levier 1 : Class weights dans la CrossEntropyLoss

Au lieu de traiter toutes les classes de la même façon, on surpondère les classes critiques. Une erreur sur le cancer coûte plus cher qu'une erreur sur l'adipose.

### Levier 2 : Seuil de décision

Au lieu de prendre la classe avec la plus haute probabilité (argmax), on baisse le seuil pour le cancer. Si le modèle donne ne serait-ce que 30% de probabilité cancer, on le signale au pathologiste.

Ces deux techniques sont complémentaires :
- Les class weights agissent **pendant l'entraînement** — le modèle apprend à être plus prudent
- Le seuil de décision agit **après l'entraînement** — on change l'interprétation des prédictions

---


In [ ]:
# BONUS — Démonstration : class weights + seuil de décision
# Sur le meilleur modèle (à adapter après comparaison)

# 1. Class weights
train_labels_all = train_dataset.labels.flatten()
class_counts = np.bincount(train_labels_all)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_weights)
class_weights[8] *= 2.0  # cancer : un faux négatif peut coûter une vie
class_weights[6] *= 1.5  # mucosa : souvent confondu avec cancer

print("=== Class weights (surpondération clinique) ===")
for c in range(9):
    print(f"  {c}: {CLASS_NAMES[c]:<40s} poids = {class_weights[c]:.4f}")

# 2. Seuil de décision (démonstration conceptuelle)
print("
=== Seuil de décision ===")
print("  Seuil standard : argmax (classe avec le plus haut score)")
print("  Seuil clinique : si P(cancer) > 0.3, signaler au pathologiste")
print("  Conséquence : plus de faux positifs, mais moins de faux négatifs")
print("  Trade-off : précision ↓ mais recall cancer ↑")


In [ ]:
# Temps total du notebook
notebook_total_time = time.time() - notebook_start_time
print(f"Temps total du notebook : {notebook_total_time:.1f}s ({notebook_total_time/60:.1f} min)")
